In [ ]:
# ========== 依赖安装（Google Colab / CUDA 环境）==========
# 安装带 CUDA 12.4 的 PyTorch 生态：torch / torchvision / torchaudio（版本钉死，避免和后续库冲突）
!pip install -q --upgrade torch==2.5.1+cu124 torchvision==0.20.1+cu124 torchaudio==2.5.1+cu124 --index-url https://download.pytorch.org/whl/cu124
# 安装推理与量化相关库：requests、bitsandbytes（4bit）、transformers、accelerate、openai
!pip install -q requests bitsandbytes==0.46.0 transformers==4.48.3 accelerate==1.3.0 openai


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 标准库 os：读环境变量等
import os
# requests：HTTP 请求（本笔记本后面未必直接用到，但与原依赖一致）
import requests
# IPython 展示工具：Markdown / display / update_display（流式或富文本展示用）
from IPython.display import Markdown, display, update_display
# OpenAI 官方客户端：也可用于兼容 OpenAI 协议的云端 API
from openai import OpenAI
# Google Colab：挂载 Drive（导入保留；本笔记本当前格未调用）
from google.colab import drive
# Hugging Face Hub 登录：拉取受控模型（如 Llama）需要 token
from huggingface_hub import login
# Colab Secrets：userdata.get('KEY') 读取密钥，避免写进代码
from google.colab import userdata
# transformers：分词器、因果语言模型、流式解码、量化配置
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
# PyTorch：张量与 GPU 计算后端
import torch


In [ ]:
# ========== 模型常量：Hugging Face 上的 Llama 3.1 Instruct ==========
# 模型 id 字符串必须保持原样，才能从 Hub 正确下载权重
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"


In [ ]:
# ========== Hugging Face 登录 ==========
# 从 Colab Secrets 读取 HF_TOKEN（需事先在 Colab 里配置）
hf_token = userdata.get('HF_TOKEN')
# 登录 Hub；add_to_git_credential=True 便于后续 git/Hub 凭证复用
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== OpenAI 客户端（可选备用；本笔记本主路径是本地 Llama）==========
# 从 Colab Secrets 读取 OPENAI_API_KEY
openai_api_key = userdata.get('OPENAI_API_KEY')
# 用该 key 构造 OpenAI 客户端实例
openai = OpenAI(api_key=openai_api_key)


In [ ]:
# ========== 对话消息：system 定角色，user 给字段规格 ==========
# system prompt：要求模型直接产出 JSON 合成测试数据（字符串内容勿改，影响模型行为）
system_message = "You are an assistant that produces synthetic test data. The fields, data type of the field like numeric, date, alphanumeric etc., will be provided. Generate data considering all cases, if it is a workflow audit data then consider all touchpoint movements. Do not provide a python script to generate the data. Provide the data as a json with arrays."
# user prompt：列出列名与类型约束（ID / TRACKING_ID / 日期 / IN SCOPE 等）
user_prompt = """Create a synthetic dataset for testing. 
Column names and type - 
ID: 10 digit number
TRACKING_ID: 13 character alphanumeric
CASE REPORT DATE : DD-MMM-YYYY HH:MM:SS
NOTIFICATION DATE : DD-MMM-YYYY HH:MM:SS
IN SCOPE : (Yes/No)
"""

# Chat Completions 风格的 messages 列表：稍后交给 chat template
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]


In [ ]:
# ========== 4bit 量化配置（BitsAndBytes / QLoRA 同款思路）==========
# BitsAndBytesConfig：把大模型权重压到约 4bit，降低 VRAM
quant_config = BitsAndBytesConfig(
    # 启用 4bit 加载
    load_in_4bit=True,
    # 双重量化：进一步压缩量化常数本身
    bnb_4bit_use_double_quant=True,
    # 计算用 bfloat16（需 GPU 支持；与原代码一致）
    bnb_4bit_compute_dtype=torch.bfloat16,
    # 量化类型 NF4：NormalFloat4，常用于 LLM 权重量化
    bnb_4bit_quant_type="nf4"
)


In [ ]:
# ========== 加载 Llama + 流式生成合成数据 ==========
# 从 Hub 加载与 LLAMA 匹配的分词器（Tokenizer）
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
# 填充 token 设为 eos，避免 pad_token 缺失导致 generate 报错
tokenizer.pad_token = tokenizer.eos_token
# 把 chat messages 套进模型对话模板，得到张量并放到 CUDA
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
# TextStreamer：边生成边把 token 解码打印到输出
streamer = TextStreamer(tokenizer)
# 以 4bit + device_map=auto 加载因果语言模型
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
# 生成最多 2000 个新 token；streamer 负责流式展示
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)
